# Module 2: Crop Mapping with Deep Learning


In [ ]:
# Core scientific computing
import numpy as np
import json
import pandas as pd

# Geospatial data handling
import rasterio
from rasterio.warp import calculate_default_transform, reproject, transform_geom
from rasterio.mask import mask
from rasterio.enums import Resampling
import pystac_client
from shapely.geometry import box, Polygon, mapping
from shapely.ops import unary_union
import geopandas as gpd
from rasterio.features import shapes

# Visualization
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
import folium

# Traditional image processing (for SLIC and Felzenszwalb)
from PIL import Image
import io
import base64
from skimage.segmentation import slic, mark_boundaries, felzenszwalb
from skimage.filters import sobel
from skimage.color import rgb2gray
from skimage.measure import find_contours
from scipy import ndimage

# Deep learning with PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Check if GPU is available for faster training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {device}")
device

### 2.1. Define Study Area

#### Hungarian Agricultural Region

In [ ]:
# Format: [min_lon, min_lat, max_lon, max_lat]
bbox = [18.235958657555724, 45.94301148728216, 
        18.287133906898454, 45.979438803640754]

#### Interactive Map Visualization

In [ ]:
# Extract bounding box coordinates
min_lon, min_lat, max_lon, max_lat = bbox

# Calculate center point for map positioning
center_lat = (min_lat + max_lat) / 2
center_lon = (min_lon + max_lon) / 2
print(f"✓ Map center: {center_lat:.4f}°N, {center_lon:.4f}°E")

# Create interactive map with satellite basemap
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri World Imagery'
)

# Define bounding box corners (clockwise from bottom-left)
bbox_coords = [
    [min_lat, min_lon],  # Bottom-left
    [min_lat, max_lon],  # Bottom-right
    [max_lat, max_lon],  # Top-right
    [max_lat, min_lon],  # Top-left
    [min_lat, min_lon]   # Close the polygon
]

# Add red bounding box overlay to show our study area
folium.Polygon(
    locations=bbox_coords[:-1],
    color='red',
    fill=True,
    fillColor='red',
    fillOpacity=0.2,
    weight=3
).add_to(m)

m

### 2.2. Download and Explore Sentinel-2 Data

**KEY CONCEPTS:**
- **Sentinel-2** = European satellite that captures 10m resolution multispectral imagery every 5 days
- **Temporal snapshots** = comparing the same location at different dates to detect changes
- **Cloud cover filter** = we only want scenes with <1% clouds for clear views


#### Configuration


In [ ]:
# Date ranges for two agricultural periods
date1_start = "2025-04-01"  # Early growing season
date1_end = "2025-04-30"
date2_start = "2025-07-01"  # Mid growing season
date2_end = "2025-07-31"

#### Query AWS Earth Search Catalog


**KEY CONCEPTS:**
- **STAC API** = SpatioTemporal Asset Catalog, a standardized way to search satellite imagery
- **AWS** = Amazon Web Services hosts petabytes of public satellite data for free
- **Cloud cover threshold** = filtering for <1% clouds ensures clear imagery


In [ ]:
# Connect to AWS Earth Search STAC API
print("\nConnecting to AWS Earth Search...")
catalog = pystac_client.Client.open("https://earth-search.aws.element84.com/v1")
print("✓ Connected to catalog")

In [ ]:
# Search for Sentinel-2 scenes in Period 1 (April)
print("\nSearching for cloud-free imagery...")

search1 = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=f"{date1_start}/{date1_end}",
    query={"eo:cloud_cover": {"lt": 1}}  # Less than 1% cloud cover
)
items1 = list(search1.items())
print(f"✓ Period 1 (April): Found {len(items1)} scenes")


In [ ]:
# Search for Sentinel-2 scenes in Period 2 (July)
search2 = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=f"{date2_start}/{date2_end}",
    query={"eo:cloud_cover": {"lt": 1}}
)
items2 = list(search2.items())
print(f"✓ Period 2 (July): Found {len(items2)} scenes")

In [ ]:
# Select the first scene from each period
item1 = items1[0]
item2 = items2[0]

print("\nSelected scenes:")
print(f"  April: {item1.datetime.date()}")
print(f"  July: {item2.datetime.date()}")

In [ ]:
item1

#### Download RGB Bands


In [ ]:
def download_rgb(item, bbox, output_filename):
    """Download and save RGB bands from Sentinel-2 scene"""
    print(f"\nDownloading {item.datetime.date()}...")
    
    # Get URLs for Red, Green, Blue bands
    red_url = item.assets['red'].href
    green_url = item.assets['green'].href
    blue_url = item.assets['blue'].href
    
    bands = []
    
    # Download and process each band
    for band_name, url in [('Red', red_url), ('Green', green_url), ('Blue', blue_url)]:
        with rasterio.open(url) as src:
            # Create bounding box geometry
            bbox_geom = box(bbox[0], bbox[1], bbox[2], bbox[3])
            bbox_transformed = transform_geom('EPSG:4326', src.crs, mapping(bbox_geom))
            
            # Crop to bounding box
            out_image, out_transform = mask(src, [bbox_transformed], crop=True)
            out_meta = src.meta.copy()
            
            # Update metadata for cropped image
            out_meta.update({
                "driver": "GTiff",
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform
            })
            
            # Reproject to standard lat/lon (EPSG:4326)
            dst_crs = 'EPSG:4326'
            transform, width, height = calculate_default_transform(
                src.crs, dst_crs, out_image.shape[2], out_image.shape[1],
                *rasterio.transform.array_bounds(out_image.shape[1], out_image.shape[2], out_transform)
            )
            
            out_meta.update({
                'crs': dst_crs,
                'transform': transform,
                'width': width,
                'height': height
            })
            
            # Perform reprojection
            reprojected = np.zeros((height, width), dtype=out_image.dtype)
            reproject(
                source=out_image[0],
                destination=reprojected,
                src_transform=out_transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                resampling=Resampling.bilinear
            )
            
            bands.append(reprojected)
            print(f"  ✓ {band_name}: {reprojected.shape}")
    
    # Stack bands into RGB image (C, H, W)
    rgb = np.stack([bands[0], bands[1], bands[2]], axis=0)
    
    # Save as GeoTIFF
    out_meta['count'] = 3
    with rasterio.open(output_filename, 'w', **out_meta) as dst:
        dst.write(rgb)
    
    print(f"  ✓ Saved: {output_filename}")
    
    # Return as (H, W, C) for visualization
    return np.transpose(rgb, (1, 2, 0))

In [ ]:
# Download both dates
if items1 and items2:
    rgb1 = download_rgb(item1, bbox, 'sentinel2_april.tif')
    rgb2 = download_rgb(item2, bbox, 'sentinel2_july.tif')
    
    print("\n" + "=" * 60)
    print("✓ Download complete!")
    print(f"  April: sentinel2_april.tif - Shape: {rgb1.shape}")
    print(f"  July: sentinel2_july.tif - Shape: {rgb2.shape}")
    print("=" * 60)

#### Normalize to Physical Reflectance

**KEY CONCEPTS:**
- **Normalization** = converting integer values to physically meaningful reflectance (0-1)
- **Reflectance** = fraction of sunlight reflected by the surface (0 = black, 1 = perfect mirror)
- **Percentile stretch** = contrast enhancement that ignores extreme outliers for better visualization


In [ ]:
def preprocess_sentinel2(input_file, output_file):
    """
    Normalize and enhance Sentinel-2 imagery for analysis and visualization
    
    Steps:
    1. Load raw GeoTIFF
    2. Normalize to [0, 1] physical reflectance
    3. Enhance contrast using 2-98 percentile stretch
    4. Save as float32 GeoTIFF
    """
    print(f"\nProcessing {input_file}...")
    
    # Load raw data
    with rasterio.open(input_file) as src:
        rgb = np.transpose(src.read(), (1, 2, 0))
        metadata = src.meta.copy()
    
    print(f"  Raw range: [{rgb.min()}, {rgb.max()}]")
    
    # Normalize to physical reflectance [0, 1]
    normalized = rgb.astype(np.float32)
    scale = 10000 if normalized.max() > 255 else 255
    normalized = normalized / scale
    normalized = np.clip(normalized, 0, 1)
    
    print(f"  Normalized (÷{scale}): [{normalized.min():.3f}, {normalized.max():.3f}]")
    
    # Enhance contrast (2-98 percentile stretch per band)
    enhanced = normalized.copy()
    for c in range(3):
        p2, p98 = np.percentile(enhanced[:, :, c], (2, 98))
        if p98 > p2:
            enhanced[:, :, c] = np.clip((enhanced[:, :, c] - p2) / (p98 - p2), 0, 1)
    
    print(f"  Enhanced: [{enhanced.min():.3f}, {enhanced.max():.3f}]")
    
    # Save as float32 GeoTIFF
    img_export = np.transpose(enhanced, (2, 0, 1))
    metadata.update({'dtype': 'float32', 'count': 3, 'nodata': None})
    
    with rasterio.open(output_file, 'w', **metadata) as dst:
        dst.write(img_export.astype(np.float32))
    
    print(f"  ✓ Saved: {output_file}")
    
    return enhanced

In [ ]:
# Process both images
image_april = preprocess_sentinel2('sentinel2_april.tif', 'april_processed.tif')
image_july = preprocess_sentinel2('sentinel2_july.tif', 'july_processed.tif')

#### Visualize Temporal Comparison


In [ ]:
# Load processed images for visualization
with rasterio.open('april_processed.tif') as src:
    rgb_april_vis = np.transpose(src.read(), (1, 2, 0))
    
with rasterio.open('july_processed.tif') as src:
    rgb_july_vis = np.transpose(src.read(), (1, 2, 0))

In [ ]:
# Create side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(rgb_april_vis)
axes[0].set_title('April 2025\nEarly Growing Season', 
                  fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(rgb_july_vis)
axes[1].set_title('July 2025\nMid Growing Season', 
                  fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.suptitle('Sentinel-2 RGB Temporal Comparison', 
             fontsize=16, fontweight='bold')
plt.tight_layout()

### 2.3. Color Thresholding

**KEY CONCEPTS:**
- **Color thresholding** = classifying pixels using hand-crafted rules based on RGB values
- **Vegetation index** = ratio comparing green vs red channels (healthy plants reflect more green)
- **Rule-based classification** = if-then logic without any machine learning


#### Calculate Spectral Indices

- **NDVI proxy** = (Green - Red) / (Green + Red), approximates true NDVI using only RGB
- **Brightness** = average reflectance across all bands
- **Spectral signature** = unique color/reflectance pattern of different surface types



In [ ]:
# Load July image (peak growing season)
with rasterio.open('july_processed.tif') as src:
    image = np.transpose(src.read(), (1, 2, 0))  # (H, W, 3)

print(f"✓ Loaded image: {image.shape}")

In [ ]:
# Extract individual color channels
red = image[:, :, 0]
green = image[:, :, 1]
blue = image[:, :, 2]

# Calculate vegetation index (approximates NDVI using RGB only)
ndvi_proxy = (green - red) / (green + red + 1e-6)

# Calculate overall brightness
brightness = image.mean(axis=2)

print("✓ Calculated spectral indices")
print(f"  NDVI proxy range: [{ndvi_proxy.min():.3f}, {ndvi_proxy.max():.3f}]")
print(f"  Brightness range: [{brightness.min():.3f}, {brightness.max():.3f}]")

#### Apply Classification Rules

**KEY CONCEPTS:**
- **Green crops** = Green > Red × 1.1 (healthy growing vegetation)
- **Yellow/ripe crops** = High red + green, low blue (mature wheat, harvested stubble)
- **Bare soil** = High brightness but low vegetation index

In [ ]:
# Initialize empty label map (0 = crops, 1 = bare soil, 2 = other)
labels = np.zeros(image.shape[:2], dtype=np.uint8)

# Test 1: Green crops (actively growing vegetation)
is_green = (green > red * 1.1) & (green > blue * 1.05)

# Test 2: Yellow/tan crops (ripe, harvested, or mature crops)
# Yellow = high red AND green, low blue
is_yellow = (red > 0.25) & (green > 0.25) & (blue < red * 0.85) & (~is_green)

is_yellow

In [ ]:
# Rule 1: Green OR Yellow = Crops (class 0)
labels[is_green | is_yellow] = 0

# Rule 2: Bright but not vegetated = Bare Soil (class 1)
labels[(~is_green) & (~is_yellow) & (brightness > 0.35)] = 1

# Rule 3: Dark pixels = Other - water, shadows, buildings (class 2)
labels[(~is_green) & (~is_yellow) & (brightness <= 0.35)] = 2

In [ ]:
# Define class names
class_names = ['Crops (Green + Ripe)', 'Bare Soil', 'Other (Water/Infrastructure)']

# Calculate and display class distribution
print("\nClassification results:")
for i in range(3):
    count = np.sum(labels == i)
    percentage = count / labels.size * 100
    print(f"  {class_names[i]}: {count:,} pixels ({percentage:.1f}%)")

#### Visualize Results


In [ ]:
image_vis = image.copy()

for c in range(3):
    p2, p98 = np.percentile(image_vis[:, :, c], (2, 98))
    if p98 > p2:
        image_vis[:, :, c] = np.clip((image_vis[:, :, c] - p2) / (p98 - p2), 0, 1)

# Create three-panel comparison
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Panel 1: Original RGB
axes[0].imshow(image_vis)
axes[0].set_title('Original Sentinel-2 RGB', fontsize=12, fontweight='bold')
axes[0].axis('off')

# Panel 2: Classification map
colors = ['#2ecc71', '#d4a574', '#3498db']  # Green, Brown, Blue
cmap = plt.matplotlib.colors.ListedColormap(colors)
axes[1].imshow(labels, cmap=cmap, vmin=0, vmax=2)
axes[1].set_title('Color Thresholding Result\n(Rule-Based Classification)', 
                  fontsize=12, fontweight='bold')
axes[1].axis('off')

# Panel 3: RGB with semi-transparent overlay
axes[2].imshow(image_vis)
overlay = np.zeros((*labels.shape, 4))
overlay[labels == 0] = [0.18, 0.8, 0.44, 0.4]   # Green (crops)
overlay[labels == 1] = [0.83, 0.65, 0.45, 0.4]  # Brown (soil)
overlay[labels == 2] = [0.2, 0.6, 0.86, 0.4]    # Blue (other)
axes[2].imshow(overlay)
axes[2].set_title('RGB + Classification Overlay', fontsize=12, fontweight='bold')
axes[2].axis('off')

# Add legend
legend_elements = [
    Patch(facecolor='#2ecc71', label='Crops'),
    Patch(facecolor='#d4a574', label='Bare Soil'),
    Patch(facecolor='#3498db', label='Other')
]
axes[1].legend(handles=legend_elements, loc='upper right', fontsize=10)

plt.suptitle('Color Thresholding Baseline (No Machine Learning)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()

### 2.4. SLIC Superpixel Segmentation - Simple Linear Iterative Clustering

**KEY CONCEPTS:**
- **SLIC** = Simple Linear Iterative Clustering algorithm
- **Superpixel** = group of similar, connected pixels forming a parcel
- **Unsupervised** = no manual labels needed, algorithm finds boundaries automatically


#### Run SLIC Segmentation

**KEY CONCEPTS:**
- **n_segments** = target number of parcels to create
- **compactness** = balance between color similarity and spatial regularity
- **sigma** = Gaussian smoothing before segmentation to reduce noise
- **Adaptive clustering** = final parcel count varies based on actual field structure


In [ ]:
# Load July image
with rasterio.open('july_processed.tif') as src:
    image = np.transpose(src.read(), (1, 2, 0))

In [ ]:
# SLIC segmentation parameters
n_segments = 200      # Target number of field parcels
                      # n_segments=50   - fewer, larger parcels
                      # n_segments=250  - many smaller parce

compactness = 5      # Balance: boundary following vs regular shapes
sigma = 2             # Gaussian smoothing before segmentation

print("Running SLIC segmentation...")
print(f"  n_segments:  {n_segments}")
print(f"  compactness: {compactness}")
print(f"  sigma:       {sigma}")

In [ ]:
# Run SLIC algorithm
segments_slic = slic(
    image,
    n_segments=n_segments,
    compactness=compactness,
    sigma=sigma,
    start_label=1
)

n_parcels = len(np.unique(segments_slic))
print(f"\n✓ SLIC complete: {n_parcels} parcels detected")

# Calculate parcel size statistics
parcel_ids, parcel_sizes = np.unique(segments_slic, return_counts=True)
pixel_area_ha = 0.01  # 10m × 10m Sentinel-2 pixel = 0.01 hectares

print(f"\nParcel size statistics:")
print(f"  Mean:   {np.mean(parcel_sizes):.0f} pixels2 ({np.mean(parcel_sizes) * pixel_area_ha:.2f} ha)")
print(f"  Median: {np.median(parcel_sizes):.0f} pixels2 ({np.median(parcel_sizes) * pixel_area_ha:.2f} ha)")
print(f"  Range:  {np.min(parcel_sizes):.0f} - {np.max(parcel_sizes):.0f} pixels2")

#### Visualize Segmentation Results


In [ ]:
# Enhance image for visualization
image_vis = image.copy()
for c in range(3):
    p2, p98 = np.percentile(image_vis[:, :, c], (2, 98))
    if p98 > p2:
        image_vis[:, :, c] = np.clip((image_vis[:, :, c] - p2) / (p98 - p2), 0, 1)

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Original RGB
axes[0].imshow(image_vis)
axes[0].set_title('Original Sentinel-2 RGB', fontsize=13, fontweight='bold')
axes[0].axis('off')

# SLIC parcels with boundaries
axes[1].imshow(mark_boundaries(image_vis, segments_slic, color=(1, 1, 0), mode='thick'))
axes[1].set_title(f'SLIC Segmentation\n({n_parcels} detected parcels)', 
                  fontsize=13, fontweight='bold')
axes[1].axis('off')

plt.suptitle('Field Parcel Detection with SLIC', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

#### Export Field Parcels for Module 3


In [ ]:
with rasterio.open('july_processed.tif') as src:
    transform = src.transform
    crs = src.crs

geometries = []
properties = []

# Export ALL parcels (not just crops)
for parcel_id in np.unique(segments_slic):
    mask = (segments_slic == parcel_id).astype(np.uint8)
    
    # Extract polygon from raster mask
    for geom, val in shapes(mask, transform=transform):
        if val == 1:  # Parcel pixels
            # Calculate parcel area
            parcel_pixels = mask.sum()
            area_ha = parcel_pixels * 0.01  # 10m pixels = 0.01 ha
            
            geometries.append(Polygon(geom['coordinates'][0]))
            properties.append({
                'parcel_id': int(parcel_id),
                'area_ha': round(area_ha, 2),
                'pixels': int(parcel_pixels)
            })
            break 


In [ ]:
# Create GeoDataFrame
gdf_parcels = gpd.GeoDataFrame(properties, geometry=geometries, crs=crs)

# Save to GeoJSON
gdf_parcels.to_file('field_parcels.geojson', driver='GeoJSON')

In [ ]:
print(f"✓ Exported {len(gdf_parcels)} field parcels")
print(f"  File: field_parcels.geojson")
print(f"  Total parcels: {len(gdf_parcels)}")
print(f"  Total area: {gdf_parcels['area_ha'].sum():.1f} hectares")
print(f"\n  Sample parcels:")
print(gdf_parcels.head(3))


#### Classify Parcels as Cropland

In [ ]:
# Calculate NDVI for the entire scene
red = image[:, :, 0]
green = image[:, :, 1]
blue = image[:, :, 2]
ndvi = (green - red) / (green + red + 1e-6)
brightness = image.mean(axis=2)

In [ ]:
# Extract features for each parcel
parcel_features = []

for parcel_id in np.unique(segments_slic):
    mask = segments_slic == parcel_id
    
    features = {
        'parcel_id': parcel_id,
        'mean_red': red[mask].mean(),
        'mean_green': green[mask].mean(),
        'mean_blue': blue[mask].mean(),
        'mean_ndvi': ndvi[mask].mean(),
        'std_ndvi': ndvi[mask].std(),
        'mean_brightness': brightness[mask].mean(),
        'size_pixels': mask.sum()
    }
    
    parcel_features.append(features)

df_parcels = pd.DataFrame(parcel_features)
df_parcels.head(3)

In [ ]:
df_parcels['is_green_crop'] = (
    (df_parcels['mean_green'] > df_parcels['mean_red'] * 1.1) & 
    (df_parcels['mean_green'] > df_parcels['mean_blue'] * 1.05)
)

df_parcels['is_yellow_crop'] = (
    (df_parcels['mean_red'] > 0.25) & 
    (df_parcels['mean_green'] > 0.25) & 
    (df_parcels['mean_blue'] < df_parcels['mean_red'] * 0.85) &
    (~df_parcels['is_green_crop'])
)

# Combine crop types and add filters
df_parcels['is_crop'] = df_parcels['is_green_crop'] | df_parcels['is_yellow_crop']
df_parcels['is_crop'] = df_parcels['is_crop'] & (df_parcels['size_pixels'] > 50)
df_parcels['is_crop'] = df_parcels['is_crop'] & (df_parcels['mean_ndvi'] > 0.1)

# Statistics
n_crop_parcels = df_parcels['is_crop'].sum()
n_total_parcels = len(df_parcels)

In [ ]:
print(f"  ✓ Classification complete")
print(f"    Crop parcels: {n_crop_parcels} ({n_crop_parcels/n_total_parcels*100:.1f}%)")
print(f"    Non-crop: {n_total_parcels - n_crop_parcels}")

In [ ]:
crop_mask_slic = np.zeros_like(segments_slic, dtype=np.uint8)

for _, row in df_parcels[df_parcels['is_crop']].iterrows():
    crop_mask_slic[segments_slic == row['parcel_id']] = 1

crop_pixels_slic = crop_mask_slic.sum()


#### Visualize Classification Results


In [ ]:
# Create visualization
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Original RGB
axes[0].imshow(image_vis)
axes[0].set_title('Sentinel-2 (RGB)', fontsize=12, fontweight='bold')
axes[0].axis('off')

# Detected crop parcels (colored by NDVI)
crop_parcels_vis = np.zeros_like(segments_slic, dtype=float)
for _, row in df_parcels[df_parcels['is_crop']].iterrows():
    crop_parcels_vis[segments_slic == row['parcel_id']] = row['mean_ndvi']

axes[1].imshow(crop_parcels_vis, cmap='RdYlGn', vmin=0, vmax=0.8)
axes[1].set_title(f'Detected Crop Parcels\n({n_crop_parcels} parcels - mean NDVI)', 
                  fontsize=12, fontweight='bold')
axes[1].axis('off')

# RGB with green crop overlay
overlay = image_vis.copy()
overlay[crop_mask_slic == 1] = overlay[crop_mask_slic == 1] * 0.5 + np.array([0, 1, 0]) * 0.5
axes[2].imshow(overlay)
axes[2].set_title('RGB + Detected Crops (Binary)', fontsize=12, fontweight='bold')
axes[2].axis('off')

plt.suptitle('SLIC Parcel-Based Crop Classification', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("SLIC CLASSIFICATION SUMMARY")
print("="*60)
print(f"✓ Total parcels: {n_total_parcels}")
print(f"✓ Crop parcels: {n_crop_parcels} ({n_crop_parcels/n_total_parcels*100:.1f}%)")
print(f"✓ Crop area: {crop_pixels_slic * pixel_area_ha:.1f} hectares")

### 2.5. U-Net Deep Learning

**KEY CONCEPTS:**
- **U-Net** = convolutional neural network architecture for semantic segmentation
- **Pixel-by-pixel classification** = predict a label for every single pixel independently
- **Deep learning** = model learns features automatically from training data
- **Supervised learning** = requires labeled training examples (input images + ground truth masks)


#### Prepare 4-Channel Input

In [ ]:
# Load July image (crops are mature and spectrally distinct)
with rasterio.open('july_processed.tif') as src:
    image_for_dl = np.transpose(src.read(), (1, 2, 0))

In [ ]:
# Extract RGB channels (already normalized to [0, 1])
red = image_for_dl[:, :, 0]
green = image_for_dl[:, :, 1]
blue = image_for_dl[:, :, 2]

# Calculate NDVI proxy (green-red approximation)
ndvi_proxy = (green - red) / (green + red + 1e-6)
ndvi_proxy = np.clip(ndvi_proxy, -1, 1)

# Normalize NDVI to [0, 1] for neural network
ndvi_normalized = (ndvi_proxy + 1) / 2

In [ ]:
# Stack into 4-channel input: [Red, Green, Blue, NDVI]
# PyTorch expects (C, H, W) format
input_4ch = np.stack([red, green, blue, ndvi_normalized], axis=0)

print(f"✓ Created 4-channel input")
print(f"  Shape: {input_4ch.shape} (Channels, Height, Width)")

In [ ]:
# Visualize the 4 channels
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(red, cmap='Reds')
axes[0].set_title('Channel 1: Red', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(green, cmap='Greens')
axes[1].set_title('Channel 2: Green', fontweight='bold')
axes[1].axis('off')

axes[2].imshow(blue, cmap='Blues')
axes[2].set_title('Channel 3: Blue', fontweight='bold')
axes[2].axis('off')

axes[3].imshow(ndvi_normalized, cmap='RdYlGn', vmin=0, vmax=1)
axes[3].set_title('Channel 4: NDVI (normalized)', fontweight='bold')
axes[3].axis('off')

plt.suptitle('4-Channel Input to U-Net', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

#### Create Training Labels

In [ ]:
# Convert 3-class labels to binary
labels_all = (labels == 0).astype(np.float32)  # 1=crops, 0=non-crop

# Physical spatial split
C, H, W = input_4ch.shape
split_col = W // 2


input_4ch_train = input_4ch[:, :, :split_col].copy()  # Left half
labels_train_cut = labels_all[:, :split_col].copy()

input_4ch_test = input_4ch[:, :, split_col:].copy()   # Right half
labels_test_cut = labels_all[:, split_col:].copy()

print(f"✓ Binary labels created & spatially split")
print(f"  Train: {labels_train_cut.sum():,.0f} crop pixels (left {split_col} cols)")
print(f"  Test:  {labels_test_cut.sum():,.0f} crop pixels (right {W-split_col} cols)")

In [ ]:
# Visualize the split in one row
fig, axes = plt.subplots(1, 4, figsize=(12, 5))

# Training RGB
rgb_train = input_4ch_train[:3, :, :].transpose(1, 2, 0)
axes[0].imshow(rgb_train)
axes[0].set_title('Training RGB\n(LEFT HALF)', fontweight='bold', fontsize=11, color='blue')
axes[0].axis('off')

# Training Labels
axes[1].imshow(labels_train_cut, cmap='RdYlGn', vmin=0, vmax=1)
axes[1].set_title(f'Training Labels\n{labels_train_cut.sum():,.0f} pixels', 
                  fontweight='bold', fontsize=11, color='blue')
axes[1].axis('off')

# Test RGB
rgb_test = input_4ch_test[:3, :, :].transpose(1, 2, 0)
axes[2].imshow(rgb_test)
axes[2].set_title('Test RGB\n(RIGHT - NEVER SEEN!)', fontweight='bold', fontsize=11, color='red')
axes[2].axis('off')

# Test Labels
axes[3].imshow(labels_test_cut, cmap='RdYlGn', vmin=0, vmax=1)
axes[3].set_title(f'Test Labels\n{labels_test_cut.sum():,.0f} pixels', 
                  fontweight='bold', fontsize=11, color='red')
axes[3].axis('off')

plt.suptitle('Spatial Split: Training (Blue) vs Test (Red) - No Overlap!', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

#### Creating a PyTorch Dataset Class


In [ ]:
class CropDataset(Dataset):
    """
    PyTorch Dataset for extracting random patches from satellite imagery.
    """
    def __init__(self, images, labels, patch_size=128, num_patches=150, train_region='full'):
        """
        Args:
            images: (C, H, W) - 4-channel input (RGB + NDVI)
            labels: (H, W) - binary labels (crop=1, non-crop=0)
            patch_size: Size of square patches to extract
            num_patches: Number of random patches to generate
            train_region: 'full' to use entire input (already cropped to left half)
        """
        self.images = images
        self.labels = labels
        self.patch_size = patch_size
        self.num_patches = num_patches
        
        # Get valid sampling region
        C, H, W = images.shape
        self.H = H
        self.W = W
        
        # Pre-generate random patch locations
        self.patch_locations = []
        for _ in range(num_patches):
            # Random top-left corner (ensure patch fits)
            y = np.random.randint(0, H - patch_size + 1)
            x = np.random.randint(0, W - patch_size + 1)
            self.patch_locations.append((y, x))
    
    def __len__(self):
        return self.num_patches
    
    def __getitem__(self, idx):
        # Get pre-generated location
        y, x = self.patch_locations[idx]
        
        # Extract patch
        image_patch = self.images[:, y:y+self.patch_size, x:x+self.patch_size]
        label_patch = self.labels[y:y+self.patch_size, x:x+self.patch_size]
        
        # Convert to PyTorch tensors
        image_tensor = torch.from_numpy(image_patch).float()
        label_tensor = torch.from_numpy(label_patch).float().unsqueeze(0)  # Add channel dim
        
        return image_tensor, label_tensor

print("✓ CropDataset class defined")

#### Creating Training Patches


In [ ]:
# Create dataset from left half only
dataset = CropDataset(
    input_4ch_train,       # Left half images (4 channels)
    labels_train_cut,      # Left half labels (binary)
    patch_size=128,        # 128×128 pixel patches
    num_patches=150,       # Random samples from left half
    train_region='full'    # Use entire left half (already cropped)
)

print(f"✓ Training dataset created")
print(f"  Patches: {len(dataset)}")
print(f"  Patch size: 128×128 pixels")
print(f"  Source: LEFT HALF only (right half never touched!)")
print(f"\n  → Network will learn from 150 diverse crop examples")
print(f"  → Each patch shows different crop patterns/boundaries")
print(f"  → Right half remains completely unseen for testing")

In [ ]:
# Visualize sample patches to confirm they're from left half
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(8):
    img_patch, label_patch = dataset[i]
    
    ax = axes[i//4, i%4]
    
    # Show RGB
    rgb = img_patch[:3].permute(1, 2, 0).numpy()
    ax.imshow(rgb)
    
    # Show label overlay
    label_viz = label_patch.squeeze().numpy()
    overlay = np.zeros((*label_viz.shape, 4))
    overlay[label_viz == 1] = [0, 1, 0, 0.4]  # Green for crops
    ax.imshow(overlay)
    
    # Get patch location
    y, x = dataset.patch_locations[i]
    ax.set_title(f'Patch {i}: x={x} (left half)', fontsize=10)
    ax.axis('off')

plt.suptitle('Training Patches (All from LEFT HALF)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Split into train (80%) and validation (20%)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Create data loaders
batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"✓ Data split completed")
print(f"  Training patches: {len(train_dataset)}")
print(f"  Validation patches: {len(val_dataset)}")
print(f"  Batch size: {batch_size}")

#### Defining U-Net Model


In [ ]:
model = nn.Sequential(
    # ENCODER (Downsampling)
    nn.Conv2d(4, 16, kernel_size=3, padding=1),  # 4 → 16 channels
    nn.ReLU(),
    nn.Conv2d(16, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),  # 128×128 → 64×64

    nn.Conv2d(16, 32, kernel_size=3, padding=1),  # 16 → 32 channels
    nn.ReLU(),
    nn.Conv2d(32, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),  # 64×64 → 32×32

    # BOTTLENECK
    nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 32 → 64 channels
    nn.ReLU(),
    nn.Conv2d(64, 64, kernel_size=3, padding=1),
    nn.ReLU(),

    # DECODER (Upsampling)
    nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),  # 32×32 → 64×64
    nn.ReLU(),
    nn.Conv2d(32, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    
    nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2),  # 64×64 → 128×128
    nn.ReLU(),
    nn.Conv2d(16, 16, kernel_size=3, padding=1),
    nn.ReLU(),

    # OUTPUT
    nn.Conv2d(16, 1, kernel_size=1),  # 16 → 1 channel (crop probability)
    nn.Sigmoid()
)

model = model.to(device)


total_params = sum(p.numel() for p in model.parameters())

print(f"✓ U-Net created")
print(f"  Architecture: Simplified U-Net (no skip connections)")
print(f"  Input: 4 channels (Red, Green, Blue, NDVI)")
print(f"  Output: 1 channel (crop probability [0, 1] per pixel)")
print(f"  Parameters: {total_params:,}")
print(f"  Device: {device}")

print(f"\n  Training strategy:")
print(f"    • Standard Binary Cross-Entropy loss")
print(f"    • Complete binary labels: 1=crop, 0=non-crop")
print(f"    • Training on LEFT HALF only (physically split)")
print(f"    • RIGHT HALF held out for testing generalization")

#### Train the Model

In [ ]:
# Setup training with STANDARD BCE (no masking needed!)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("✓ Training setup:")
print("  Loss: Standard Binary Cross-Entropy")
print("  Optimizer: Adam (lr=0.001)")
print("  Complete binary labels: 1=crop, 0=non-crop")


# Intersection over Union
# Standard IoU calculation (no masking needed)
def calculate_iou(pred, target, threshold=0.5):
    """Calculate standard IoU for binary segmentation"""
    pred_binary = (pred > threshold).float()
    target_binary = target.float()
    
    intersection = (pred_binary * target_binary).sum()
    union = pred_binary.sum() + target_binary.sum() - intersection
    
    if union == 0:
        return 1.0 if intersection == 0 else 0.0
    
    iou = intersection / union
    return iou.item()

print("✓ IoU metric defined")

In [ ]:
NUM_EPOCHS = 15

history = {
    'train_loss': [],
    'val_loss': [],
    'train_iou': [],
    'val_iou': []
}

best_val_iou = 0.0


for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print("-"*60)
    
    # Training phase
    model.train()
    train_loss = 0.0
    train_iou_sum = 0.0

    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        
        # Standard BCE loss (all pixels have labels now)
        loss = criterion(outputs, masks)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_iou_sum += calculate_iou(outputs.detach().cpu(), masks.cpu())

    avg_train_loss = train_loss / len(train_loader)
    avg_train_iou = train_iou_sum / len(train_loader)


    # Validation phase
    model.eval()
    val_loss = 0.0
    val_iou_sum = 0.0
    
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            val_loss += loss.item()
            val_iou_sum += calculate_iou(outputs.cpu(), masks.cpu())
    
    avg_val_loss = val_loss / len(val_loader)
    avg_val_iou = val_iou_sum / len(val_loader)


    # Save history
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['train_iou'].append(avg_train_iou)
    history['val_iou'].append(avg_val_iou)
    
    # Check improvement
    improved = ""
    if avg_val_iou > best_val_iou:
        best_val_iou = avg_val_iou
        improved = " ⭐ BEST"
    
    # Print progress
    print(f"  Train Loss: {avg_train_loss:.4f}  |  Train IoU: {avg_train_iou:.4f}")
    print(f"  Val Loss:   {avg_val_loss:.4f}  |  Val IoU:   {avg_val_iou:.4f}{improved}")


print(f"  Best validation IoU: {best_val_iou:.4f}")
print(f"  Trained on LEFT HALF only")
print(f"  Ready to test on RIGHT HALF (unseen data)")    

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss curve
axes[0].plot(history['train_loss'], label='Training Loss', marker='o', linewidth=2)
axes[0].plot(history['val_loss'], label='Validation Loss', marker='s', linewidth=2)
axes[0].set_xlabel('Epoch', fontweight='bold')
axes[0].set_ylabel('Loss (Standard BCE)', fontweight='bold')
axes[0].set_title('Training and Validation Loss\n(LEFT HALF data)', fontweight='bold', fontsize=11)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# IoU curve
axes[1].plot(history['train_iou'], label='Training IoU', marker='o', linewidth=2)
axes[1].plot(history['val_iou'], label='Validation IoU', marker='s', linewidth=2)
axes[1].axhline(y=best_val_iou, color='red', linestyle='--', 
                label=f'Best Val IoU: {best_val_iou:.3f}', alpha=0.7)
axes[1].set_xlabel('Epoch', fontweight='bold')
axes[1].set_ylabel('IoU Score', fontweight='bold')
axes[1].set_title('Training and Validation IoU\n(LEFT HALF data)', fontweight='bold', fontsize=11)
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, 1])

plt.suptitle('U-Net Training History (Trained on LEFT HALF Only)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Training Summary:")
print(f"  • Final training IoU: {history['train_iou'][-1]:.4f}")
print(f"  • Final validation IoU: {history['val_iou'][-1]:.4f}")
print(f"  • Best validation IoU: {best_val_iou:.4f}")
print(f"\n  ⚠️  These metrics are on LEFT HALF data only")
print(f"      Next: Test on RIGHT HALF (unseen) for true generalization!")

#### Apply U-Net to Full Image


In [ ]:
model.eval()

# Convert FULL image to tensor (both halves)
input_tensor = torch.from_numpy(input_4ch).unsqueeze(0).float().to(device)

print(f"  Input shape: {input_tensor.shape}")
print(f"  Processing entire scene (LEFT trained, RIGHT unseen)...")

with torch.no_grad():
    prediction = model(input_tensor)

# Convert to numpy
unet_prediction = prediction.cpu().squeeze().numpy()

print(f"\n✓ Prediction complete")
print(f"  Output shape: {unet_prediction.shape}")
print(f"  Value range: [{unet_prediction.min():.3f}, {unet_prediction.max():.3f}]")

# Threshold to binary
unet_binary = (unet_prediction > 0.5).astype(np.uint8)

# Split predictions by half
split_col = W // 2
pred_left = unet_binary[:, :split_col]
pred_right = unet_binary[:, split_col:]

print(f"\n✓ Binary prediction created (threshold=0.5)")
print(f"  LEFT HALF (trained):  {pred_left.sum():,} pixels ({pred_left.sum()/pred_left.size*100:.1f}%)")
print(f"  RIGHT HALF (unseen):  {pred_right.sum():,} pixels ({pred_right.sum()/pred_right.size*100:.1f}%)")
print(f"  Total:                {unet_binary.sum():,} pixels")

print(f"\n  Training was on LEFT HALF only ({labels_train_cut.sum():,} crop pixels)")
print(f"  RIGHT HALF was completely unseen during training!")
print(f"  → Can the network generalize? Let's compare to ground truth...")

In [ ]:
# Simplified visualization - 3 panels only
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# FIRST: Ensure dimensions match
if unet_binary.shape != labels_all.shape:
    min_h = min(unet_binary.shape[0], labels_all.shape[0])
    min_w = min(unet_binary.shape[1], labels_all.shape[1])
    unet_binary_adj = unet_binary[:min_h, :min_w]
    labels_all_adj = labels_all[:min_h, :min_w]
    H_adj, W_adj = min_h, min_w
    split_col_adj = W_adj // 2
else:
    unet_binary_adj = unet_binary
    labels_all_adj = labels_all
    H_adj, W_adj = labels_all.shape
    split_col_adj = W_adj // 2

# Panel 1: Original RGB with split line
axes[0].imshow(image_for_dl)
axes[0].axvline(x=split_col, color='red', linewidth=3, linestyle='--')
axes[0].set_title('RGB Image (July)\nRed line = train/test split', fontsize=12, fontweight='bold')
axes[0].axis('off')

# Panel 2: U-Net Probability Map
axes[1].imshow(unet_prediction, cmap='RdYlGn', vmin=0, vmax=1)
axes[1].axvline(x=split_col, color='white', linewidth=2, linestyle='--')
axes[1].set_title('U-Net Probability Map\n(0=non-crop, 1=crop)', 
                  fontsize=12, fontweight='bold')
axes[1].axis('off')

# Panel 3: U-Net Binary Prediction
axes[2].imshow(unet_binary_adj, cmap='RdYlGn', vmin=0, vmax=1)
axes[2].axvline(x=split_col_adj, color='white', linewidth=2, linestyle='--')
axes[2].set_title(f'U-Net Binary Prediction\n{unet_binary_adj.sum():,} pixels (threshold=0.5)', 
                  fontsize=12, fontweight='bold')
axes[2].axis('off')

# Calculate RIGHT HALF metrics for title
pred_right = unet_binary_adj[:, split_col_adj:]
truth_right = labels_all_adj[:, split_col_adj:]

from sklearn.metrics import jaccard_score
right_iou = jaccard_score(truth_right.flatten(), pred_right.flatten())

plt.suptitle(f'U-Net Crop Detection Results | Training IoU={best_val_iou:.3f} | Test IoU (RIGHT)={right_iou:.3f}', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Detailed metrics printout
print("\n" + "="*60)
print("U-NET GENERALIZATION TEST RESULTS")
print("="*60)

pred_right = unet_binary_adj[:, split_col_adj:]
truth_right = labels_all_adj[:, split_col_adj:]

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(truth_right.flatten(), pred_right.flatten())
precision = precision_score(truth_right.flatten(), pred_right.flatten(), zero_division=0)
recall = recall_score(truth_right.flatten(), pred_right.flatten(), zero_division=0)
f1 = f1_score(truth_right.flatten(), pred_right.flatten(), zero_division=0)
right_iou = jaccard_score(truth_right.flatten(), pred_right.flatten())

print(f"\n📊 LEFT HALF (Training):")
print(f"  Ground truth crops: {labels_train_cut.sum():,} pixels")
print(f"  Best validation IoU: {best_val_iou:.4f}")

print(f"\n📊 RIGHT HALF (Unseen - TRUE TEST):")
print(f"  Ground truth crops: {truth_right.sum():,} pixels")
print(f"  Predicted crops:    {pred_right.sum():,} pixels")
print(f"\n  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f} (of predicted crops, how many are real?)")
print(f"  Recall:    {recall:.4f} (of real crops, how many found?)")
print(f"  F1 Score:  {f1:.4f}")
print(f"  IoU:       {right_iou:.4f} ⭐ KEY METRIC")

print(f"\n💡 GENERALIZATION VERDICT:")
if right_iou > 0.5:
    print(f"  ✅ EXCELLENT - IoU > 0.5 on unseen data!")
    print(f"     Network successfully learned spatial patterns")
elif right_iou > 0.3:
    print(f"  ✓ GOOD - IoU > 0.3 shows decent generalization")
    print(f"    Network learned some patterns, room for improvement")
else:
    print(f"  ⚠️ POOR - IoU < 0.3 suggests overfitting or insufficient learning")
    print(f"    May need more training data, epochs, or architecture changes")

print("="*60)

### 6. Compare All Methods


In [ ]:
# Get geospatial bounds
with rasterio.open('july_processed.tif') as src:
    bounds = src.bounds
    transform = src.transform

min_lon, min_lat = bounds.left, bounds.bottom
max_lon, max_lat = bounds.right, bounds.top
center = [(min_lat + max_lat) / 2, (min_lon + max_lon) / 2]

# Create base map
m = folium.Map(
    location=center,
    zoom_start=14,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri World Imagery'
)

def array_to_base64_png(data, colormap=None):
    """Convert numpy array to base64-encoded PNG"""
    if data.ndim == 2:  # Single channel
        if colormap == 'RdYlGn':
            from matplotlib import cm
            data_colored = (cm.RdYlGn(data)[:, :, :3] * 255).astype(np.uint8)
            img = Image.fromarray(data_colored)
        else:
            img = Image.fromarray((data * 255).astype(np.uint8), mode='L')
    else:  # RGB
        img = Image.fromarray((np.clip(data, 0, 1) * 255).astype(np.uint8))
    
    buffer = io.BytesIO()
    img.save(buffer, format='PNG')
    img_str = base64.b64encode(buffer.getvalue()).decode()
    return f'data:image/png;base64,{img_str}'

# Layer 1: RGB basemap
rgb_base64 = array_to_base64_png(image_for_dl)
folium.raster_layers.ImageOverlay(
    image=rgb_base64,
    bounds=[[min_lat, min_lon], [max_lat, max_lon]],
    opacity=0.8,
    name='Sentinel-2 RGB',
    show=True
).add_to(m)

# Layer 2: Color Thresholding (from Section 3)
# Recreate if needed
color_threshold_binary = labels_all  # Using the full cleaned labels from color thresholding
color_overlay = np.zeros((*color_threshold_binary.shape, 4), dtype=np.uint8)
color_overlay[color_threshold_binary == 1] = [255, 165, 0, 180]  # Orange

img_color = Image.fromarray(color_overlay, mode='RGBA')
buffer = io.BytesIO()
img_color.save(buffer, format='PNG')
color_base64 = base64.b64encode(buffer.getvalue()).decode()

folium.raster_layers.ImageOverlay(
    image=f'data:image/png;base64,{color_base64}',
    bounds=[[min_lat, min_lon], [max_lat, max_lon]],
    opacity=0.6,
    name='Method 1: Color Thresholding',
    show=False
).add_to(m)

# Layer 3: SLIC Parcels
slic_overlay = np.zeros((*crop_mask_slic.shape, 4), dtype=np.uint8)
slic_overlay[crop_mask_slic == 1] = [255, 255, 0, 180]  # Yellow

img_slic = Image.fromarray(slic_overlay, mode='RGBA')
buffer = io.BytesIO()
img_slic.save(buffer, format='PNG')
slic_base64 = base64.b64encode(buffer.getvalue()).decode()

folium.raster_layers.ImageOverlay(
    image=f'data:image/png;base64,{slic_base64}',
    bounds=[[min_lat, min_lon], [max_lat, max_lon]],
    opacity=0.6,
    name='Method 2: SLIC Parcels',
    show=True
).add_to(m)

# Layer 4: U-Net Prediction
unet_overlay = np.zeros((*unet_binary.shape, 4), dtype=np.uint8)
unet_overlay[unet_binary == 1] = [0, 255, 0, 180]  # Green

img_unet = Image.fromarray(unet_overlay, mode='RGBA')
buffer = io.BytesIO()
img_unet.save(buffer, format='PNG')
unet_base64 = base64.b64encode(buffer.getvalue()).decode()

folium.raster_layers.ImageOverlay(
    image=f'data:image/png;base64,{unet_base64}',
    bounds=[[min_lat, min_lon], [max_lat, max_lon]],
    opacity=0.6,
    name='Method 3: U-Net Deep Learning',
    show=False
).add_to(m)

# Add layer control (top right)
folium.LayerControl(position='topright', collapsed=False).add_to(m)


m.get_root().html.add_child(folium.Element(legend_html))

m